# Treinamento e Exportação TinyML v2 (ESP32) — Sistema de Irrigação Inteligente (PIBEX 2026 - IFPE Paulista)

Este notebook implementa a **Versão 2 (v2)** do pipeline oficial de Inteligência Artificial Embarcada (**TinyML**), reestruturada a partir das conclusões da auditoria forense e do benchmark experimental de 10 arquiteturas documentados em [`docs/AUDITORIA_TINYML_E_COMPARATIVO.md`](file:///home/victor/projects/sistema-de-irrigacao-inteligente/docs/AUDITORIA_TINYML_E_COMPARATIVO.md).

### Síntese das 9 Correções e Melhorias da v2 (`MLP-Orig` v1 $\rightarrow$ `PhysMLP-27` v2):

| # | Categoria Auditada | Problema Identificado na v1 | Solução Implementada na v2 |
| :-: | :--- | :--- | :--- |
| **1** | **Sintaxe Python** | `{=*75}` em f-string causava `SyntaxError` fatal na célula de avaliação | Uso de `sep = '=' * 75` compatível com qualquer versão do Python |
| **2** | **Data Leakage (Linhas)** | `128` duplicatas exatas (`256` linhas envolvidas) vazavam entre Treino, Validação e Teste | `df.drop_duplicates()` obrigatório pré-split (`16.411` $\rightarrow$ `16.283` linhas únicas) |
| **3** | **Sensor-Tuple Leakage** | `91,85%` das `2.886` tuplas físicas `(MOI, temp, humidity)` do Teste já estavam no Treino | Auditoria de sobreposição + avaliação dupla: **Split Estratificado** e **GroupSplit Estrito (0% Leakage)** |
| **4** | **Validação Distorcida** | `is_training=True` no `val_df` multiplicava hortaliças por 4x e filtrava Classe 2 (`MOI >= 60`) | Validação limpa e simétrica nas 8 máscaras reais sem filtragem artificial |
| **5** | **Colapso da Classe 2** | `class_weight='balanced'` (`~5.05` na Classe 2) + dropout assimétrico derrubava a Precisão C2 para `34,25%` (`ECE = 0,0676`) | Pesos calibrados por varredura de Pareto `{0: 1.0, 1: 1.15, 2: 2.20}` (`ECE = 0,0145`) |
| **6** | **Ponto Cego Modo 6** | Apenas 4 de $2^3=8$ máscaras treinadas; Modo 6 (`Apenas Cultura [0,C,0]`) caía para `83,27%` | **Full-8 Mask Context Dropout** cobrindo todas as 8 combinações (`89,28%` no Modo 6) |
| **7** | **Física Psicrométrica** | Apenas 3 leituras lineares brutas (`temp`, `humidity`, `MOI`) + 20 one-hot (`23` features) | **Engenharia Psicrométrica (`PhysMLP-27`)**: `+4` features não-lineares (`VPD_kPa`, `EvapDemand`, `Temp/MOI`, `MOI*Temp`) |
| **8** | **Falsa Quantização** | `Optimize.DEFAULT` sem `representative_dataset` mantinha 100% dos pesos em `Float32` (`7.620 B`) | **Full Integer Quantization (`INT8`)** real com `representative_dataset` nos 8 modos (`5.400 B`, `-33,6%` Flash, `~99%` concordância) |
| **9** | **Sentinelas de Falha** | Falha do DHT22 (`-128 °C` e `255%`) sofria `clip` para `0 °C` e `100%` (frio/úmido = `OFF`) | Validação explícita `isValidSensorReading()` em Python e C++ inline (`-128`, `-999`, `255`, `NaN`) antes do `clamp` |

## 1. Preparação do Ambiente e Reprodutibilidade Científica

In [ ]:
import os
import glob
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Sementes fixas para total reprodutibilidade científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow Version : {tf.__version__}")
print(f"NumPy Version      : {np.__version__}")
print(f"Pandas Version     : {pd.__version__}")

## 2. Carregamento do Dataset, Deduplicação Pré-Split e Auditoria Forense de Tuplas

- Carrega `cropdata_updated.csv` localmente ou via `kagglehub` (`chaitanyagopidesi/smart-agriculture-dataset`).
- Remove imediatamente as **128 duplicatas exatas** (`16.411` $\rightarrow$ `16.283` registros únicos) antes de qualquer divisão.
- Audita o número de tuplas físicas únicas `(MOI, temp, humidity)` (`2.886` tuplas).

In [ ]:
LOCAL_PATHS = [
    "datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "ai-model/datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "cropdata_updated.csv",
]

dataset_path = next((p for p in LOCAL_PATHS if os.path.exists(p)), None)
if dataset_path is None:
    import kagglehub
    path = kagglehub.dataset_download("chaitanyagopidesi/smart-agriculture-dataset")
    csv_files = glob.glob(os.path.join(path, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {path}")
    dataset_path = csv_files[0]

df_raw = pd.read_csv(dataset_path)
n_raw = len(df_raw)
n_dup = int(df_raw.duplicated().sum())

# Correção #2 da Auditoria: Deduplicação obrigatória antes do split
df = df_raw.drop_duplicates().reset_index(drop=True)
n_clean = len(df)
n_unique_tuples = int(df.groupby(["MOI", "temp", "humidity"]).ngroups)

print(f"Dataset carregado de   : {dataset_path}")
print(f"Registros brutos       : {n_raw}")
print(f"Duplicatas removidas   : {n_dup} -> {n_clean} registros únicos")
print(f"Tuplas físicas únicas  : {n_unique_tuples} combinações (MOI, temp, humidity) | Média: {n_clean/n_unique_tuples:.2f}x por tupla")
display(df.head())

## 3. Engenharia Psicrométrica (`PhysMLP-27`), Proteção de Sentinelas e Codificação

1. **Sensores Físicos Normalizados (`[0..2]`):** `temp` ($[0..50\text{ °C}]$), `humidity` ($[0..100\%]$), `MOI` ($[0..100\%]$).
2. **Contexto Agronômico Opcional via *One-Hot* (`[3..22]`):**
   - `soil_type` (`7` posições, `0 = UNKNOWN` $\rightarrow$ vetor nulo)
   - `crop ID` (`5` posições, `0 = UNKNOWN` $\rightarrow$ vetor nulo)
   - `Seedling Stage` (`8` posições, `0 = UNKNOWN` $\rightarrow$ vetor nulo)
3. **4 Features Psicrométricas e Hidro-Térmicas (`[23..26]`):**
   - `vec[23]` — **Déficit de Pressão de Vapor ($VPD$, Equação de Tetens):**
     $$e_s(T) = 0.61078 \cdot \exp\left(\frac{17.27 \cdot T}{T + 237.3}\right), \quad VPD = e_s(T) \cdot \left(1 - \frac{RH}{100}\right), \quad VPD_{\text{norm}} = \text{clip}\left(\frac{VPD}{8.0}, 0, 1\right)$$
   - `vec[24]` — **Demanda Evaporativa Termo-Higrométrica:** $\text{clip}\left(\frac{T \cdot (100 - RH)}{4000.0}, 0, 1\right)$
   - `vec[25]` — **Razão Calor / Umidade do Solo:** $\frac{\text{clip}(T_{\text{norm}} / (MOI_{\text{norm}} + 0.10), 0, 5)}{5.0}$
   - `vec[26]` — **Acoplamento Hidro-Térmico (Assinatura da Classe 2):** $MOI_{\text{norm}} \cdot T_{\text{norm}}$
4. **Validação de Sentinelas de Hardware (`SensorsManager`):** Rejeita explicitamente `-128` (`int8_t`), `-999` (`int16_t`), `255` (`uint8_t`) e `NaN` antes da normalização.

In [ ]:
# Limites explícitos de normalização compartilhados entre Python e ESP32 (C++)
TEMP_MIN, TEMP_MAX = 0.0, 50.0
HUM_MIN, HUM_MAX = 0.0, 100.0
SOIL_MIN, SOIL_MAX = 0.0, 100.0

# Escalas psicrométricas compartilhadas com o C++
VPD_MAX_KPA = 8.0
EVAP_DEMAND_DIVISOR = 4000.0
RATIO_MOI_OFFSET = 0.10
RATIO_MAX_CLIP = 5.0

# Sentinelas de erro do hardware (SensorsManager.cpp) para proteção contra falha silenciosa
SENTINEL_TEMP_ERR_I8 = -128
SENTINEL_TEMP_ERR_I16 = -999
SENTINEL_HUM_ERR_U8 = 255

# Limiares operacionais de segurança
DEFAULT_CONFIDENCE_THRESHOLD = 0.55
THERMAL_STRESS_SOIL_CAP_PCT = 65
THERMAL_STRESS_TEMP_MIN_C = 32

SOIL_TYPES = [
    "Clay Soil", "Sandy Soil", "Red Soil", "Loam Soil",
    "Black Soil", "Alluvial Soil", "Chalky Soil",
]
CROP_TYPES = ["Tomato", "Chilli", "Carrot", "Potato", "Wheat"]
SEEDLING_STAGES = [
    "Germination", "Seedling Stage",
    "Vegetative Growth / Root or Tuber Development",
    "Flowering", "Pollination", "Fruit/Grain/Bulb Formation",
    "Maturation", "Harvest",
]

SOIL_TO_ID = {name: i + 1 for i, name in enumerate(SOIL_TYPES)}
CROP_TO_ID = {name: i + 1 for i, name in enumerate(CROP_TYPES)}
STAGE_TO_ID = {name: i + 1 for i, name in enumerate(SEEDLING_STAGES)}

BASE_DIM = 3 + len(SOIL_TYPES) + len(CROP_TYPES) + len(SEEDLING_STAGES)  # 23
INPUT_DIM = BASE_DIM + 4                                                 # 27 (PhysMLP-27)
NUM_CLASSES = 3

def is_valid_sensor_reading(temp, hum, moi):
    """Rejeita sentinelas de erro do DHT22/HW-080 (-128, -999, 255, NaN) antes do clip."""
    if temp is None or hum is None or moi is None:
        return False
    if np.isnan(temp) or np.isnan(hum) or np.isnan(moi):
        return False
    if int(round(temp)) in (SENTINEL_TEMP_ERR_I8, SENTINEL_TEMP_ERR_I16):
        return False
    if int(round(hum)) == SENTINEL_HUM_ERR_U8 or int(round(moi)) == SENTINEL_HUM_ERR_U8:
        return False
    if not (-10.0 <= float(temp) <= 65.0):
        return False
    if not (0.0 <= float(hum) <= 100.0) or not (0.0 <= float(moi) <= 100.0):
        return False
    return True

def compute_vpd_kpa(temp_c, rh_pct):
    es = 0.61078 * np.exp((17.27 * temp_c) / (temp_c + 237.3))
    return es * np.clip(1.0 - rh_pct / 100.0, 0.0, 1.0)

def encode_features(temp, hum, moi, soil_id=0, crop_id=0, stage_id=0, validate_sentinels=True):
    if validate_sentinels and not is_valid_sensor_reading(temp, hum, moi):
        raise ValueError(f"Sentinela de falha de sensor detectado: T={temp}, H={hum}, MOI={moi}")

    vec = np.zeros(INPUT_DIM, dtype=np.float32)
    t_n = float(np.clip((temp - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0.0, 1.0))
    h_n = float(np.clip((hum - HUM_MIN) / (HUM_MAX - HUM_MIN), 0.0, 1.0))
    m_n = float(np.clip((moi - SOIL_MIN) / (SOIL_MAX - SOIL_MIN), 0.0, 1.0))
    vec[0], vec[1], vec[2] = t_n, h_n, m_n

    if 1 <= int(soil_id) <= len(SOIL_TYPES):
        vec[3 + (int(soil_id) - 1)] = 1.0
    if 1 <= int(crop_id) <= len(CROP_TYPES):
        vec[3 + len(SOIL_TYPES) + (int(crop_id) - 1)] = 1.0
    if 1 <= int(stage_id) <= len(SEEDLING_STAGES):
        vec[3 + len(SOIL_TYPES) + len(CROP_TYPES) + (int(stage_id) - 1)] = 1.0

    vpd = float(compute_vpd_kpa(temp, hum))
    vec[23] = float(np.clip(vpd / VPD_MAX_KPA, 0.0, 1.0))
    vec[24] = float(np.clip((temp * (100.0 - hum)) / EVAP_DEMAND_DIVISOR, 0.0, 1.0))
    vec[25] = float(np.clip(t_n / (m_n + RATIO_MOI_OFFSET), 0.0, RATIO_MAX_CLIP) / RATIO_MAX_CLIP)
    vec[26] = float(m_n * t_n)
    return vec

def encode_batch(temp, hum, moi, soil_id, crop_id, stage_id):
    n = len(temp)
    X = np.zeros((n, INPUT_DIM), dtype=np.float32)
    t_n = np.clip((temp - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0.0, 1.0)
    h_n = np.clip((hum - HUM_MIN) / (HUM_MAX - HUM_MIN), 0.0, 1.0)
    m_n = np.clip((moi - SOIL_MIN) / (SOIL_MAX - SOIL_MIN), 0.0, 1.0)
    X[:, 0], X[:, 1], X[:, 2] = t_n, h_n, m_n

    s_valid = (soil_id >= 1) & (soil_id <= len(SOIL_TYPES))
    X[s_valid, 3 + (soil_id[s_valid] - 1)] = 1.0
    c_valid = (crop_id >= 1) & (crop_id <= len(CROP_TYPES))
    X[c_valid, 3 + len(SOIL_TYPES) + (crop_id[c_valid] - 1)] = 1.0
    st_valid = (stage_id >= 1) & (stage_id <= len(SEEDLING_STAGES))
    X[st_valid, 3 + len(SOIL_TYPES) + len(CROP_TYPES) + (stage_id[st_valid] - 1)] = 1.0

    vpd = compute_vpd_kpa(temp, hum)
    X[:, 23] = np.clip(vpd / VPD_MAX_KPA, 0.0, 1.0)
    X[:, 24] = np.clip((temp * (100.0 - hum)) / EVAP_DEMAND_DIVISOR, 0.0, 1.0)
    X[:, 25] = np.clip(t_n / (m_n + RATIO_MOI_OFFSET), 0.0, RATIO_MAX_CLIP) / RATIO_MAX_CLIP
    X[:, 26] = m_n * t_n
    return X

print(f"Dimensão do vetor PhysMLP-27 v2: {INPUT_DIM} features ({BASE_DIM} base + 4 psicrométricas)")

## 4. Divisão Dupla (Estratificada Limpa + GroupSplit Zero-Leakage) e *Full-8 Mask Context Dropout*

- **Correção #3:** Além do split estratificado (70/15/15), construímos o **`GroupSplit` por `(MOI, temp, humidity)`** com **0,00% de sobreposição de tuplas de sensores** entre Treino, Validação e Teste.
- **Correção #4 e #6:** *Full-8 Mask Context Dropout* cobrindo todas as $2^3 = 8$ combinações de máscara `(Soil, Crop, Stage)`, incluindo o Modo 6 (`Apenas Cultura [0, C, 0]`), sem distorcer a validação.
- **Correção #5:** Pesos de classe calibrados `{0: 1.0, 1: 1.15, 2: 2.20}`.

In [ ]:
ALL_8_MASKS = [
    ("M1: Completo [S,C,St]", (1, 1, 1)),
    ("M2: Solo+Cultura [S,C,0]", (1, 1, 0)),
    ("M3: Solo+Estágio [S,0,St]", (1, 0, 1)),
    ("M4: Cultura+Estágio [0,C,St]", (0, 1, 1)),
    ("M5: Apenas Solo [S,0,0]", (1, 0, 0)),
    ("M6: Apenas Cultura [0,C,0]", (0, 1, 0)),
    ("M7: Apenas Estágio [0,0,St]", (0, 0, 1)),
    ("M8: Zero Contexto [0,0,0]", (0, 0, 0)),
]

def stratified_split(dataframe, strat_col, test_size=0.30, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for _, grp in dataframe.groupby(strat_col, sort=True):
        idx = grp.index.to_numpy().copy()
        rng.shuffle(idx)
        n_test = max(1, int(round(len(idx) * test_size)))
        test_idx.append(idx[:n_test])
        train_idx.append(idx[n_test:])
    return dataframe.loc[np.concatenate(train_idx)].copy(), dataframe.loc[np.concatenate(test_idx)].copy()

def group_split_by_sensors(dataframe, seed=SEED):
    """GroupSplit estrito por tupla (MOI, temp, humidity) -> 0% de vazamento de sensores."""
    rng = np.random.default_rng(seed)
    keys = dataframe[["MOI", "temp", "humidity"]].drop_duplicates().to_numpy()
    idx = np.arange(len(keys))
    rng.shuffle(idx)
    n_train = int(0.70 * len(idx))
    n_val = int(0.15 * len(idx))
    train_keys = set(map(tuple, keys[idx[:n_train]]))
    val_keys = set(map(tuple, keys[idx[n_train:n_train + n_val]]))
    test_keys = set(map(tuple, keys[idx[n_train + n_val:]]))
    tuples = list(zip(dataframe["MOI"], dataframe["temp"], dataframe["humidity"]))
    return (
        dataframe[[t in train_keys for t in tuples]].copy(),
        dataframe[[t in val_keys for t in tuples]].copy(),
        dataframe[[t in test_keys for t in tuples]].copy(),
    )

df["soil_id"] = df["soil_type"].map(SOIL_TO_ID)
df["crop_id"] = df["crop ID"].map(CROP_TO_ID)
df["stage_id"] = df["Seedling Stage"].map(STAGE_TO_ID)
df["strat_key"] = df["crop ID"].astype(str) + "_" + df["result"].astype(str)

train_df, temp_df = stratified_split(df, "strat_key", test_size=0.30, seed=SEED)
val_df, test_df = stratified_split(temp_df, "strat_key", test_size=0.50, seed=SEED)
tr_grp, va_grp, te_grp = group_split_by_sensors(df, seed=SEED)

def build_full8_dataset(dataframe):
    t = dataframe["temp"].to_numpy(dtype=np.float32)
    h = dataframe["humidity"].to_numpy(dtype=np.float32)
    m = dataframe["MOI"].to_numpy(dtype=np.float32)
    s = dataframe["soil_id"].to_numpy(dtype=np.int32)
    c = dataframe["crop_id"].to_numpy(dtype=np.int32)
    st = dataframe["stage_id"].to_numpy(dtype=np.int32)
    y = dataframe["result"].to_numpy(dtype=np.int32)
    is_veg = (dataframe["crop ID"] != "Wheat").to_numpy()
    z = np.zeros_like(s)

    Xs, ys = [], []
    for ms, mc, mst in [(1, 1, 1), (1, 1, 0), (0, 1, 1), (0, 1, 0)]:
        Xs.append(encode_batch(t, h, m, s if ms else z, c if mc else z, st if mst else z))
        ys.append(y)
    for ms, mc, mst in [(1, 0, 1), (1, 0, 0), (0, 0, 1), (0, 0, 0)]:
        Xs.append(encode_batch(
            t[is_veg], h[is_veg], m[is_veg],
            s[is_veg] if ms else z[is_veg], z[is_veg], st[is_veg] if mst else z[is_veg]
        ))
        ys.append(y[is_veg])
    return np.vstack(Xs), np.concatenate(ys)

def build_mode_eval(dataframe, mask_tuple, veg_only_when_no_crop=True):
    ms, mc, mst = mask_tuple
    df_sub = dataframe[dataframe["crop ID"] != "Wheat"] if (veg_only_when_no_crop and mc == 0) else dataframe
    t = df_sub["temp"].to_numpy(dtype=np.float32)
    h = df_sub["humidity"].to_numpy(dtype=np.float32)
    m = df_sub["MOI"].to_numpy(dtype=np.float32)
    s = df_sub["soil_id"].to_numpy(dtype=np.int32) if ms else np.zeros(len(df_sub), dtype=np.int32)
    c = df_sub["crop_id"].to_numpy(dtype=np.int32) if mc else np.zeros(len(df_sub), dtype=np.int32)
    st = df_sub["stage_id"].to_numpy(dtype=np.int32) if mst else np.zeros(len(df_sub), dtype=np.int32)
    y = df_sub["result"].to_numpy(dtype=np.int32)
    return encode_batch(t, h, m, s, c, st), y

X_train, y_train = build_full8_dataset(train_df)
X_val, y_val = build_full8_dataset(val_df)
class_weight_dict = {0: 1.0, 1: 1.15, 2: 2.20}

print(f"Split Estratificado -> Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")
print(f"GroupSplit Estrito  -> Treino: {len(tr_grp)} | Validação: {len(va_grp)} | Teste: {len(te_grp)}")
print(f"Treino Full-8       -> {X_train.shape[0]} amostras | Validação Limpa: {X_val.shape[0]} amostras")
print(f"Pesos Calibrados v2 -> {class_weight_dict}")

## 5. Arquitetura `PhysMLP-27` v2 e Treinamento

- Utiliza apenas camadas `Dense` com ativação `ReLU` e saída `Softmax` (`AddFullyConnected`, `AddRelu`, `AddSoftmax` no TFLite Micro).
- Ocupa apenas **`5.400 bytes` (`5.27 KB`) em Flash INT8** e **`< 2.5 KB` de `tensor_arena` (SRAM)** no ESP32.

In [ ]:
def build_phys_mlp_v2(input_dim=INPUT_DIM, num_classes=NUM_CLASSES):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,), name="sensor_context_phys_input"),
        tf.keras.layers.Dense(32, activation="relu", name="dense_1"),
        tf.keras.layers.Dense(16, activation="relu", name="dense_2"),
        tf.keras.layers.Dense(num_classes, activation="softmax", name="decision_output"),
    ], name="PhysMLP_27_v2")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_phys_mlp_v2()
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=128,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

## 6. Avaliação Científica Completa nos 8 Modos de Contexto (M1 a M8) e no `GroupSplit` (0% Leakage)

In [ ]:
CLASS_NAMES = ["0: OFF (Nao Irrigar)", "1: IRRIGATE (Irrigar)", "2: THERMAL_STRESS (Alerta)"]

def compute_metrics(y_true, y_pred, y_prob=None):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    acc = float(np.trace(cm) / max(1, np.sum(cm)))
    prec, rec, f1, support = [], [], [], []
    for c in range(NUM_CLASSES):
        tp = int(cm[c, c])
        fp = int(cm[:, c].sum() - tp)
        fn = int(cm[c, :].sum() - tp)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        prec.append(float(p)); rec.append(float(r)); f1.append(float(f)); support.append(int(cm[c, :].sum()))
    ece = 0.0
    if y_prob is not None:
        conf = np.max(y_prob, axis=1)
        correct = (y_pred == y_true).astype(np.float64)
        bins = np.linspace(0.0, 1.0, 11)
        for i in range(10):
            mask = (conf >= bins[i]) & (conf <= bins[i+1]) if i == 0 else (conf > bins[i]) & (conf <= bins[i+1])
            if np.any(mask):
                ece += (np.sum(mask) / len(conf)) * abs(np.mean(correct[mask]) - np.mean(conf[mask]))
    return {
        "accuracy": acc, "macro_f1": float(np.mean(f1)), "ece": float(ece),
        "precision": prec, "recall": rec, "f1": f1, "support": support, "cm": cm
    }

sep = "=" * 78
print(f"\n{sep}\nAvaliação nos 8 Modos de Contexto (M1..M8) — Teste Deduplicado\n{sep}")
mode_metrics = {}
for mode_label, mtuple in ALL_8_MASKS:
    Xm, ym = build_mode_eval(test_df, mtuple)
    pm = model.predict(Xm, verbose=0)
    yhat = np.argmax(pm, axis=1)
    met = compute_metrics(ym, yhat, pm)
    mode_metrics[mode_label] = (ym, yhat, met)
    print(
        f"{mode_label:<30} | Acc: {met['accuracy']*100:6.2f}% | "
        f"Macro-F1: {met['macro_f1']*100:6.2f}% | "
        f"C2 Prec/Rec: {met['precision'][2]*100:5.2f}%/{met['recall'][2]*100:5.2f}% | "
        f"ECE: {met['ece']:.4f}"
    )

# Avaliação complementar sob GroupSplit Estrito (0% Sensor-Tuple Leakage)
X_tr_g, y_tr_g = build_full8_dataset(tr_grp)
X_va_g, y_va_g = build_full8_dataset(va_grp)
model_grp = build_phys_mlp_v2()
model_grp.fit(
    X_tr_g, y_tr_g, validation_data=(X_va_g, y_va_g),
    epochs=60, batch_size=128, class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=0),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=0),
    ],
    verbose=0,
)
print(f"\n{sep}\nGeneralização Estrita (GroupSplit — 0,00% Sensor-Tuple Overlap)\n{sep}")
for ml, mt in [("M1: Completo [S,C,St]", (1, 1, 1)), ("M6: Apenas Cultura [0,C,0]", (0, 1, 0)), ("M8: Zero Contexto [0,0,0]", (0, 0, 0))]:
    Xmg, ymg = build_mode_eval(te_grp, mt)
    pmg = model_grp.predict(Xmg, verbose=0)
    met_g = compute_metrics(ymg, np.argmax(pmg, axis=1), pmg)
    print(f"[GroupSplit] {ml:<30} | Acc: {met_g['accuracy']*100:6.2f}% | Macro-F1: {met_g['macro_f1']*100:6.2f}% | ECE: {met_g['ece']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history["loss"], label="Treino Loss", lw=2)
axes[0].plot(history.history["val_loss"], label="Validação Limpa Loss", lw=2)
axes[0].set_title("Evolução da Loss (PhysMLP-27 v2)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for ax, mkey, title, cmap in [
    (axes[1], "M1: Completo [S,C,St]", "Matriz de Confusão (M1: Completo)", "Greens"),
    (axes[2], "M8: Zero Contexto [0,0,0]", "Matriz de Confusão (M8: Zero Contexto)", "Blues"),
]:
    cm_arr = mode_metrics[mkey][2]["cm"]
    ax.imshow(cm_arr, cmap=cmap)
    ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(["OFF", "IRRIG", "STRESS"]); ax.set_yticklabels(["OFF", "IRRIG", "STRESS"])
    ax.set_title(title); ax.set_xlabel("Predito"); ax.set_ylabel("Real")
    for r in range(3):
        for c in range(3):
            ax.text(c, r, str(cm_arr[r, c]), ha="center", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Quantização Real Full INT8 (`.tflite`), Validação Cruzada e Exportação de `model_data.h` com Helpers C++ Inline

- Constrói `representative_dataset` estratificado cobrindo todos os **8 modos de contexto**.
- Converte para **Full Integer Quantization (`INT8`)** (`TFLITE_BUILTINS_INT8`, entrada e saída `tf.int8`), reduzindo o tamanho de `8.132 bytes` (`Float32`) para **`5.400 bytes` (`INT8`)**.
- Exporta `model_data.h` com constantes psicrométricas, parâmetros de quantização INT8 (`INPUT_SCALE`, `INPUT_ZERO_POINT`, `OUTPUT_SCALE`, `OUTPUT_ZERO_POINT`) e funções C++ inline `isValidSensorReading()`, `computeVpdKpa()` e `encodeFeatures27()`.

In [ ]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
tflite_path = os.path.join(OUTPUT_DIR, "model_int8.tflite")
header_path = os.path.join(OUTPUT_DIR, "model_data.h")

# Representative dataset estratificado nos 8 modos de contexto (400 amostras)
rep_sub_df = train_df.sample(n=min(50, len(train_df)), random_state=SEED)
X_rep = np.vstack([build_mode_eval(rep_sub_df, mt)[0] for _, mt in ALL_8_MASKS])

def representative_data_gen():
    for i in range(len(X_rep)):
        yield [X_rep[i:i + 1].astype(np.float32)]

# Conversão Real Full Integer Quantization (INT8) para TensorFlow Lite Micro
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"[TinyML v2] Modelo Full INT8 salvo: {tflite_path} ({len(tflite_model)} bytes / {len(tflite_model)/1024:.2f} KB)")

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
in_scale, in_zp = input_details["quantization"]
out_scale, out_zp = output_details["quantization"]

def predict_tflite(x_vec):
    xq = np.clip(np.round(np.expand_dims(x_vec, axis=0) / in_scale + in_zp), -128, 127).astype(np.int8)
    interpreter.set_tensor(input_details["index"], xq)
    interpreter.invoke()
    yq = interpreter.get_tensor(output_details["index"])[0]
    return (yq.astype(np.float32) - out_zp) * out_scale

X_check, _ = build_mode_eval(test_df, (1, 1, 1))
n_check = min(600, len(X_check))
p_keras_all = model.predict(X_check[:n_check], verbose=0)
p_tflite_all = np.array([predict_tflite(X_check[i]) for i in range(n_check)])
abs_diff = np.abs(p_keras_all - p_tflite_all)
agree_pct = 100.0 * float(np.mean(np.argmax(p_keras_all, axis=1) == np.argmax(p_tflite_all, axis=1)))

print(f"[Validação Cruzada INT8] Concordância: {agree_pct:.2f}% | Mean Diff: {np.mean(abs_diff):.5f} | Max Diff: {np.max(abs_diff):.5f}")

golden_cases = [
    ("Caso 1 (Sem Contexto - Solo Umido/Frio -> OFF)", 22.0, 78.0, 65.0, 0, 0, 0),
    ("Caso 2 (Sem Contexto - Solo Seco/Quente -> IRRIGATE)", 34.0, 45.0, 22.0, 0, 0, 0),
    ("Caso 3 (Sem Contexto - Calor Extremo + Solo Encharcado -> THERMAL_STRESS)", 38.0, 40.0, 75.0, 0, 0, 0),
    ("Caso 4 (Com Contexto: Tomate em Solo Argiloso + Floracao - Seco -> IRRIGATE)", 33.0, 50.0, 28.0, SOIL_TO_ID["Clay Soil"], CROP_TO_ID["Tomato"], STAGE_TO_ID["Flowering"]),
    ("Caso 5 (Com Contexto: Pimenta em Solo Arenoso + Germinacao - Calor + Umido -> THERMAL_STRESS)", 37.0, 42.0, 70.0, SOIL_TO_ID["Sandy Soil"], CROP_TO_ID["Chilli"], STAGE_TO_ID["Germination"]),
    ("Caso 6 (Modo 6 - Apenas Cultura: Tomate [0,1,0] - Solo Seco -> IRRIGATE)", 31.0, 52.0, 25.0, 0, CROP_TO_ID["Tomato"], 0),
]

golden_comments = []
for desc, t, h, m, s_id, c_id, st_id in golden_cases:
    vec = encode_features(t, h, m, s_id, c_id, st_id)
    probs = predict_tflite(vec)
    pred_cls = int(np.argmax(probs))
    vpd_val = float(compute_vpd_kpa(t, h))
    line = (
        f"// {desc}: Temp={t}C, Hum={h}%, Soil={m}%, VPD={vpd_val:.2f}kPa, "
        f"SoilID={s_id}, CropID={c_id}, StageID={st_id} => "
        f"Class={pred_cls} (P0={probs[0]:.3f}, P1={probs[1]:.3f}, P2={probs[2]:.3f})"
    )
    golden_comments.append(line)
    print(line)

hex_lines = ["  " + ", ".join(f"0x{b:02x}" for b in tflite_model[i:i+12]) for i in range(0, len(tflite_model), 12)]
hex_body = ",\n".join(hex_lines)
golden_block = "\n".join(golden_comments)

c_header = f"""#pragma once

#include <cstdint>
#include <cmath>
#include <algorithm>

// ============================================================================
// Modelo TinyML v2 Gerado Automaticamente - Sistema de Irrigacao Inteligente
// Arquitetura: PhysMLP-27 ({INPUT_DIM} inputs -> 32 ReLU -> 16 ReLU -> 3 Softmax)
// Quantizacao: Full Integer Quantization (INT8) para TensorFlow Lite Micro
// Baseado em: docs/AUDITORIA_TINYML_E_COMPARATIVO.md
// ============================================================================
// Golden Test Vectors (Validados no TFLite INT8 Interpreter):
{golden_block}
// ============================================================================

namespace TinyMLConfig {{
    constexpr uint8_t INPUT_FEATURE_COUNT = {INPUT_DIM};
    constexpr uint8_t BASE_FEATURE_COUNT = {BASE_DIM};
    constexpr uint8_t OUTPUT_CLASS_COUNT = {NUM_CLASSES};

    constexpr float TEMP_MIN = {TEMP_MIN:.1f}f;
    constexpr float TEMP_MAX = {TEMP_MAX:.1f}f;
    constexpr float HUMIDITY_MIN = {HUM_MIN:.1f}f;
    constexpr float HUMIDITY_MAX = {HUM_MAX:.1f}f;
    constexpr float SOIL_MIN = {SOIL_MIN:.1f}f;
    constexpr float SOIL_MAX = {SOIL_MAX:.1f}f;

    constexpr float VPD_MAX_KPA = {VPD_MAX_KPA:.1f}f;
    constexpr float EVAP_DEMAND_DIVISOR = {EVAP_DEMAND_DIVISOR:.1f}f;
    constexpr float RATIO_MOI_OFFSET = {RATIO_MOI_OFFSET:.2f}f;
    constexpr float RATIO_MAX_CLIP = {RATIO_MAX_CLIP:.1f}f;

    constexpr bool IS_INT8_QUANTIZED = true;
    constexpr float INPUT_SCALE = {float(in_scale):.8e}f;
    constexpr int32_t INPUT_ZERO_POINT = {int(in_zp)};
    constexpr float OUTPUT_SCALE = {float(out_scale):.8e}f;
    constexpr int32_t OUTPUT_ZERO_POINT = {int(out_zp)};

    // Sentinelas de falha do DHT22 (SensorsManager.cpp) - abortar inferencia se detectados!
    constexpr int8_t SENTINEL_TEMP_ERROR_I8 = {SENTINEL_TEMP_ERR_I8};
    constexpr int16_t SENTINEL_TEMP_ERROR_I16 = {SENTINEL_TEMP_ERR_I16};
    constexpr uint8_t SENTINEL_HUM_ERROR_U8 = {SENTINEL_HUM_ERR_U8};

    constexpr float DEFAULT_CONFIDENCE_THRESHOLD = {DEFAULT_CONFIDENCE_THRESHOLD:.2f}f;
    constexpr uint8_t THERMAL_STRESS_SOIL_CAP_PCT = {THERMAL_STRESS_SOIL_CAP_PCT};
    constexpr int8_t THERMAL_STRESS_TEMP_MIN_C = {THERMAL_STRESS_TEMP_MIN_C};

    enum class SoilType : uint8_t {{
        UNKNOWN = 0,
        CLAY_SOIL = 1,
        SANDY_SOIL = 2,
        RED_SOIL = 3,
        LOAM_SOIL = 4,
        BLACK_SOIL = 5,
        ALLUVIAL_SOIL = 6,
        CHALKY_SOIL = 7
    }};

    enum class CropType : uint8_t {{
        UNKNOWN = 0, // Fallback para Hortalicas Locais (Paulista-PE)
        TOMATO = 1,
        CHILLI = 2,
        CARROT = 3,
        POTATO = 4,
        WHEAT = 5
    }};

    enum class SeedlingStage : uint8_t {{
        UNKNOWN = 0,
        GERMINATION = 1,
        SEEDLING_STAGE = 2,
        VEGETATIVE_GROWTH = 3,
        FLOWERING = 4,
        POLLINATION = 5,
        FRUIT_FORMATION = 6,
        MATURATION = 7,
        HARVEST = 8
    }};

    inline bool isValidSensorReading(float tempC, float humPct, float soilPct) {{
        if (std::isnan(tempC) || std::isnan(humPct) || std::isnan(soilPct)) return false;
        const int tempInt = static_cast<int>(std::lround(tempC));
        const int humInt = static_cast<int>(std::lround(humPct));
        const int soilInt = static_cast<int>(std::lround(soilPct));
        if (tempInt == SENTINEL_TEMP_ERROR_I8 || tempInt == SENTINEL_TEMP_ERROR_I16) return false;
        if (humInt == SENTINEL_HUM_ERROR_U8 || soilInt == SENTINEL_HUM_ERROR_U8) return false;
        if (tempC < -10.0f || tempC > 65.0f) return false;
        if (humPct < 0.0f || humPct > 100.0f) return false;
        if (soilPct < 0.0f || soilPct > 100.0f) return false;
        return true;
    }}

    inline float computeVpdKpa(float tempC, float rhPct) {{
        const float es = 0.61078f * std::exp((17.27f * tempC) / (tempC + 237.3f));
        const float deficit = std::clamp(1.0f - (rhPct / 100.0f), 0.0f, 1.0f);
        return es * deficit;
    }}

    inline bool encodeFeatures27(
        float tempC, float humPct, float soilPct,
        uint8_t soilId, uint8_t cropId, uint8_t stageId,
        float outFeatures[INPUT_FEATURE_COUNT]
    ) {{
        if (!isValidSensorReading(tempC, humPct, soilPct)) return false;
        for (uint8_t i = 0; i < INPUT_FEATURE_COUNT; ++i) outFeatures[i] = 0.0f;
        const float tNorm = std::clamp((tempC - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0.0f, 1.0f);
        const float hNorm = std::clamp((humPct - HUMIDITY_MIN) / (HUMIDITY_MAX - HUMIDITY_MIN), 0.0f, 1.0f);
        const float mNorm = std::clamp((soilPct - SOIL_MIN) / (SOIL_MAX - SOIL_MIN), 0.0f, 1.0f);
        outFeatures[0] = tNorm; outFeatures[1] = hNorm; outFeatures[2] = mNorm;
        if (soilId >= 1 && soilId <= 7) outFeatures[3 + (soilId - 1)] = 1.0f;
        if (cropId >= 1 && cropId <= 5) outFeatures[10 + (cropId - 1)] = 1.0f;
        if (stageId >= 1 && stageId <= 8) outFeatures[15 + (stageId - 1)] = 1.0f;
        const float vpdKpa = computeVpdKpa(tempC, humPct);
        outFeatures[23] = std::clamp(vpdKpa / VPD_MAX_KPA, 0.0f, 1.0f);
        outFeatures[24] = std::clamp((tempC * (100.0f - humPct)) / EVAP_DEMAND_DIVISOR, 0.0f, 1.0f);
        outFeatures[25] = std::clamp(tNorm / (mNorm + RATIO_MOI_OFFSET), 0.0f, RATIO_MAX_CLIP) / RATIO_MAX_CLIP;
        outFeatures[26] = mNorm * tNorm;
        return true;
    }}

    inline int8_t quantizeInput(float value) {{
        const int32_t q = static_cast<int32_t>(std::lround(value / INPUT_SCALE)) + INPUT_ZERO_POINT;
        return static_cast<int8_t>(std::clamp(q, -128, 127));
    }}

    inline float dequantizeOutput(int8_t qValue) {{
        return (static_cast<float>(qValue) - static_cast<float>(OUTPUT_ZERO_POINT)) * OUTPUT_SCALE;
    }}
}}

const unsigned int g_model_len = {len(tflite_model)};
alignas(8) const unsigned char g_model_data[] = {{
{hex_body}
}};
"""

with open(header_path, "w", encoding="utf-8") as f:
    f.write(c_header)
print(f"[TinyML v2] Header C++ exportado: {header_path}")

for dest_dir in [".", "../embedded/include", "embedded/include"]:
    if os.path.isdir(dest_dir):
        dest = os.path.join(dest_dir, "model_data.h")
        with open(dest, "w", encoding="utf-8") as f:
            f.write(c_header)
        print(f"[TinyML v2] Header sincronizado: {dest}")